# Phase 2: Final ML Model Training
In this notebook, we train our final Random Forest classifier against the newly generated, objective Composite SQI Labels.

In [1]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import GroupShuffleSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

project_root = r"c:\Users\rajka\PPG-IIIT"
df = pd.read_csv(os.path.join(project_root, 'outputs', 'sqi_labeled_features.csv'))
print(f"Loaded {len(df)} records for training.")

Loaded 3888 records for training.


### Data Preparation & Grouping
We use `GroupShuffleSplit` on the Subject ID to ensure no data leakage occurs between subjects.

In [2]:
exclude_cols = ['ID', 'SQI_Label', 'Segment_ID', 'Original_ID', 'SQI_Label_New', 'sqi_score', 'SQI_Score']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols]
y = df['SQI_Label_New']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Extract Subject ID (first 3 digits of the ID)
groups = df['ID'].astype(str).str[:3]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y_encoded, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data scaled and split successfully.")

Data scaled and split successfully.


### Model Training

In [3]:
print("Training Random Forest on SQI Labels...")
rf = RandomForestClassifier(random_state=42)
rf_params = {'n_estimators': [200], 'max_depth': [20, None], 'class_weight': ['balanced']}
rf_grid = GridSearchCV(rf, rf_params, cv=3, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train_scaled, y_train)

best_rf = rf_grid.best_estimator_
rf_preds = best_rf.predict(X_test_scaled)

print("\n=== Final Composite SQI Validation ===")
print(f"Test Accuracy: {accuracy_score(y_test, rf_preds):.4f}\n")
print(classification_report(y_test, rf_preds, target_names=le.classes_))

Training Random Forest on SQI Labels...

=== Final Composite SQI Validation ===
Test Accuracy: 0.9444

              precision    recall  f1-score   support

        Good       0.95      0.97      0.96       274
    Moderate       0.93      0.93      0.93       396
        Poor       0.96      0.94      0.95       320

    accuracy                           0.94       990
   macro avg       0.95      0.95      0.95       990
weighted avg       0.94      0.94      0.94       990



### Save Final ML Pipeline
These artifacts will be used by our FastAPI backend.

In [4]:
print("\nSaving final ML artifacts...")
os.makedirs(os.path.join(project_root, 'models'), exist_ok=True)
joblib.dump(best_rf, os.path.join(project_root, 'models', 'rf_sqi_model.pkl'))
joblib.dump(scaler, os.path.join(project_root, 'models', 'sqi_scaler.pkl'))
joblib.dump(le, os.path.join(project_root, 'models', 'sqi_label_encoder.pkl'))
print("Models saved successfully!")


Saving final ML artifacts...
Models saved successfully!
